In [98]:
from pathlib import Path 

import numpy as np
import pandas as pd
import sqlalchemy as db
from tqdm.auto import tqdm
from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.http import models

In [99]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)  

In [117]:
collections = [x.name for x in CLIENT.get_collections().collections]
if 'FacialEmbeddings' not in collections:
        CLIENT.recreate_collection(collection_name='FacialEmbeddings',
                                   vectors_config=VectorParams(size=128, distance=Distance.COSINE))

/tmp/ipykernel_77541/3511449327.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  CLIENT.recreate_collection(collection_name='FacialEmbeddings',


In [102]:
username = 'amos'
password = 'M0$hicat'
host = '192.168.0.131'
port = '3306'
database = 'CineFace'

In [103]:
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)
conn = engine.connect()

In [104]:
df = pd.read_csv('/home/amos/datasets/shining_bat_encodings.csv', index_col=0)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,face_num,img_width,img_height,filename,filepath,distance_from_center,pct_of_frame,encoding,series_id,episode_id
0,479,94,695,397,535,208,634,196,589,237,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,26.63,0.0708,-0.0674933\n0.0943281\n0.0449327\n0.00756572\n...,81505,15659200
1,515,114,730,422,570,231,669,223,623,276,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,27.04,0.0715,-0.0405336\n0.0716574\n0.0567907\n0.00282796\n...,81505,15659200
2,509,108,721,410,562,226,662,216,616,264,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,26.74,0.0695,-0.0331698\n0.0957054\n0.0514582\n0.0187592\n-...,81505,15659200
3,515,99,732,407,584,220,685,212,648,261,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,27.02,0.0722,-0.0560095\n0.0887332\n0.0417233\n0.0227062\n-...,81505,15659200
4,492,117,709,416,550,233,651,220,610,268,...,0,1280,720,shining_bat.mp4,/home/amos/programs/CineFace/data/test_videos/...,26.66,0.0698,-0.0416598\n0.103417\n0.0755524\n0.000812375\n...,81505,15659200


In [105]:
def parse_vector(vector):
    return np.array([float(x) for x in vector.split('\n')])

In [111]:
encodings = df['encoding'].map(parse_vector)[:1]
CLIENT.upload_points(collection_name='FacialEmbeddings',
              points=[
              PointStruct(
                id=idx,
                vector=vector.tolist()
            )
            for idx, vector in encodings.items()]
)

In [112]:
vectors = CLIENT.retrieve('FacialEmbeddings', [0], with_vectors=True)
vectors[0]

Record(id=0, payload={}, vector=[-0.052032847, 0.07272069, 0.034640126, 0.0058326675, -0.077702396, -0.055261828, -0.052692074, -0.0118235, 0.05582276, -0.016610919, 0.18796131, -0.017875714, -0.19646315, -0.024813114, 0.003404643, 0.07639049, -0.112453, -0.036034357, -0.111439206, -0.12467075, 0.025625214, -0.00084152794, 0.0007637114, -0.012005363, -0.10696549, -0.17812188, -0.061053615, -0.09071806, 0.06566357, -0.124883525, 0.037854068, 0.021244615, -0.14760603, -0.051163778, -0.008856176, -0.0037847208, 0.010602109, -0.04266755, 0.124114126, -0.03054269, -0.096692026, -0.016740205, 0.023024624, 0.18017565, 0.16586865, 0.036311973, 0.007769854, -0.033603374, 0.041549925, -0.2262474, 0.012393066, 0.105742015, 0.03514146, 0.07467509, 0.042731535, -0.111022145, 0.014195045, 0.085499614, -0.103719085, 0.02384798, 0.026273876, -0.09161543, -0.0449105, -0.074708626, 0.15185925, 0.059875324, -0.037661877, -0.086541146, 0.1298784, -0.11914547, -0.03179669, 0.07479019, -0.05985513, -0.11028

In [113]:
parse_vector(df.at[0, 'encoding'])

array([-0.0674933 ,  0.0943281 ,  0.0449327 ,  0.00756572, -0.10079   ,
       -0.0716817 , -0.0683484 , -0.0153366 ,  0.0724093 , -0.0215465 ,
        0.24381   , -0.0231871 , -0.254838  , -0.0321858 ,  0.00441626,
        0.0990883 , -0.145866  , -0.0467412 , -0.144551  , -0.161714  ,
        0.0332392 , -0.00109157,  0.00099063, -0.0155725 , -0.138748  ,
       -0.231047  , -0.0791944 , -0.117673  ,  0.0851741 , -0.16199   ,
        0.0491016 ,  0.027557  , -0.191464  , -0.066366  , -0.0114876 ,
       -0.00490927,  0.0137523 , -0.0553453 ,  0.160992  , -0.0396178 ,
       -0.125422  , -0.0217142 ,  0.0298659 ,  0.233711  ,  0.215153  ,
        0.0471013 ,  0.0100785 , -0.0435879 ,  0.0538956 , -0.293472  ,
        0.0160754 ,  0.137161  ,  0.045583  ,  0.0968632 ,  0.0554283 ,
       -0.14401   ,  0.0184128 ,  0.110904  , -0.134537  ,  0.0309339 ,
        0.0340806 , -0.118837  , -0.0582547 , -0.0969067 ,  0.196981  ,
        0.077666  , -0.0488523 , -0.112255  ,  0.168469  , -0.15

In [114]:
encodings[0]

array([-0.0674933 ,  0.0943281 ,  0.0449327 ,  0.00756572, -0.10079   ,
       -0.0716817 , -0.0683484 , -0.0153366 ,  0.0724093 , -0.0215465 ,
        0.24381   , -0.0231871 , -0.254838  , -0.0321858 ,  0.00441626,
        0.0990883 , -0.145866  , -0.0467412 , -0.144551  , -0.161714  ,
        0.0332392 , -0.00109157,  0.00099063, -0.0155725 , -0.138748  ,
       -0.231047  , -0.0791944 , -0.117673  ,  0.0851741 , -0.16199   ,
        0.0491016 ,  0.027557  , -0.191464  , -0.066366  , -0.0114876 ,
       -0.00490927,  0.0137523 , -0.0553453 ,  0.160992  , -0.0396178 ,
       -0.125422  , -0.0217142 ,  0.0298659 ,  0.233711  ,  0.215153  ,
        0.0471013 ,  0.0100785 , -0.0435879 ,  0.0538956 , -0.293472  ,
        0.0160754 ,  0.137161  ,  0.045583  ,  0.0968632 ,  0.0554283 ,
       -0.14401   ,  0.0184128 ,  0.110904  , -0.134537  ,  0.0309339 ,
        0.0340806 , -0.118837  , -0.0582547 , -0.0969067 ,  0.196981  ,
        0.077666  , -0.0488523 , -0.112255  ,  0.168469  , -0.15

In [115]:
p = PointStruct(
                id=0,
                vector=parse_vector(df.at[0, 'encoding'])
            )

In [126]:
CLIENT.upload_points('FacialEmbeddings', [p])

In [127]:
print(CLIENT.retrieve('FacialEmbeddings', [0], with_vectors=True))

[Record(id=0, payload={}, vector=[-0.052032847, 0.07272069, 0.034640126, 0.0058326675, -0.077702396, -0.055261828, -0.052692074, -0.0118235, 0.05582276, -0.016610919, 0.18796131, -0.017875714, -0.19646315, -0.024813114, 0.003404643, 0.07639049, -0.112453, -0.036034357, -0.111439206, -0.12467075, 0.025625214, -0.00084152794, 0.0007637114, -0.012005363, -0.10696549, -0.17812188, -0.061053615, -0.09071806, 0.06566357, -0.124883525, 0.037854068, 0.021244615, -0.14760603, -0.051163778, -0.008856176, -0.0037847208, 0.010602109, -0.04266755, 0.124114126, -0.03054269, -0.096692026, -0.016740205, 0.023024624, 0.18017565, 0.16586865, 0.036311973, 0.007769854, -0.033603374, 0.041549925, -0.2262474, 0.012393066, 0.105742015, 0.03514146, 0.07467509, 0.042731535, -0.111022145, 0.014195045, 0.085499614, -0.103719085, 0.02384798, 0.026273876, -0.09161543, -0.0449105, -0.074708626, 0.15185925, 0.059875324, -0.037661877, -0.086541146, 0.1298784, -0.11914547, -0.03179669, 0.07479019, -0.05985513, -0.1102

In [122]:
p

PointStruct(id=0, vector=[-0.0674933, 0.0943281, 0.0449327, 0.00756572, -0.10079, -0.0716817, -0.0683484, -0.0153366, 0.0724093, -0.0215465, 0.24381, -0.0231871, -0.254838, -0.0321858, 0.00441626, 0.0990883, -0.145866, -0.0467412, -0.144551, -0.161714, 0.0332392, -0.00109157, 0.000990632, -0.0155725, -0.138748, -0.231047, -0.0791944, -0.117673, 0.0851741, -0.16199, 0.0491016, 0.027557, -0.191464, -0.066366, -0.0114876, -0.00490927, 0.0137523, -0.0553453, 0.160992, -0.0396178, -0.125422, -0.0217142, 0.0298659, 0.233711, 0.215153, 0.0471013, 0.0100785, -0.0435879, 0.0538956, -0.293472, 0.0160754, 0.137161, 0.045583, 0.0968632, 0.0554283, -0.14401, 0.0184128, 0.110904, -0.134537, 0.0309339, 0.0340806, -0.118837, -0.0582547, -0.0969067, 0.196981, 0.077666, -0.0488523, -0.112255, 0.168469, -0.154547, -0.0412444, 0.0970125, -0.0776398, -0.143056, -0.270979, 0.0807617, 0.382451, 0.174577, -0.14738, 0.054682, -0.0940755, -0.0327114, 0.0166876, -0.0185261, -0.072558, -0.0334414, -0.0787775, 0.1

In [129]:
(10000000 * 1.5 * 128 * 4)/1024/1024/1024

7.152557373046875